In [ ]:
import boto3
from config import *
import base64
import json

In [ ]:
print(boto3.__version__)

In [ ]:
bedrock_agent = boto3.Session().client('bedrock-agent')
bedrock_rt = boto3.Session().client('bedrock-runtime')

In [ ]:
image_path = "resources/cat_image.png"

In [ ]:
def image_to_bytes(image_path):
    with open(image_path, "rb") as image_file:
        # Read the file content
        binary_data = image_file.read()
        # Encode the binary data to base64
        base64_encoded = base64.b64encode(binary_data)
        # Convert bytes to string
        base64_string = base64_encoded.decode('utf-8')      # <---- adding this step
    return {'data': base64_string}

In [ ]:
image_bytes = image_to_bytes(image_path)
image_bytes_data = image_bytes['data']
images = [image_bytes_data]

In [ ]:
def invoke_llama(modelId):
    # Embed the prompt in Llama 3's instruction format.
    formatted_prompt = """
    <|begin_of_text|><|start_header_id|>user<|end_header_id|><|image|>Describe this image in two
    sentences
    <|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>
    """

    payload = {
        "prompt": formatted_prompt,
        "images": images,
        "max_gen_len": 2048,
        "temperature": 0.0
    }

    # Convert the payload to JSON
    request_body = json.dumps(payload)

    # Invoke the model
    response = bedrock_rt.invoke_model(
        modelId=model_id,
        body=request_body,
        contentType='application/json'
    )

    # Assuming your response is stored in a variable called 'response'
    body = response['body']
    content = body.read().decode('utf-8')

    # Parse the JSON content
    parsed_content = json.loads(content)

    print(json.dumps(parsed_content, indent=2))

In [ ]:
models_and_inference_profiles = ["us.meta.llama3-2-90b-instruct-v1:0", "meta.llama3-2-11b-instruct-v1:0", "us.meta.llama3-2-11b-instruct-v1:0"]

In [ ]:
for inference_id in models_and_inference_profiles:
    invoke_llama(inference_id)
